In [ ]:
#02_modelado.py
#Tarea 2 - Modelado de alquileres estimacion de precio mensual.

import warnings
import numpy as np
import pandas as pd
from math import radians, cos, sin, asin, sqrt

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import mlflow
import mlflow.sklearn
import mlflow.xgboost

warnings.filterwarnings("ignore")

#------Carga de dataset limpio-----------

df = pd.read_parquet("output/dataset_unificado.parquet")
print(f"Dataset cargado: {len(df)} filas, {df.shape[1]} columnas")
print(f"Distribución por fuente:\n{df['fuente'].value_counts()}")
print(f"Distribución por tipo:\n{df['tipo_inmueble'].value_counts()}")

#-----------Feature Engineering------------

# 1.1 Coordenadas de referencia (centro de CABA)
LAT_REF, LON_REF = -34.6037, -58.3816   # Obelisco

def haversine(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos puntos geográficos."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

# 1.2 Distancia al centro (solo para filas con coords válidas)
df["dist_centro_km"] = df.apply(
    lambda r: haversine(r["latitud"], r["longitud"], LAT_REF, LON_REF)
    if pd.notna(r["latitud"]) else np.nan,
    axis=1
)

# 1.3 Precio mediano del barrio (lag espacial sencillo)
#     Se calcula en el set completo y luego se usa en CV — esto es una
#     simplificación; en producción debería calcularse solo sobre train.
barrio_median = df.groupby("barrio")["precio_ars"].median().rename("barrio_precio_mediano")
df = df.join(barrio_median, on="barrio")

# 1.4 Precio mediano por m² del barrio
df["barrio_precio_m2_mediano"] = df.groupby("barrio")["precio_por_m2"].transform("median")

# 1.5 Ratio superficie cubierta / total
df["ratio_cubierta_total"] = df["superficie_cubierta_m2"] / (df["superficie_total_m2"] + 1)

# 1.6 Encode tipo_inmueble y fuente
le_tipo = LabelEncoder()
df["tipo_enc"]  = le_tipo.fit_transform(df["tipo_inmueble"])
le_fuente = LabelEncoder()
df["fuente_enc"] = le_fuente.fit_transform(df["fuente"])

# 1.7 Encode barrio (target encoding ya hecho con barrio_precio_mediano;
#     también añadimos frecuencia como proxy de zona grande/chica)
df["barrio_freq"] = df.groupby("barrio")["barrio"].transform("count")

print(f"\nFeatures creadas: dist_centro_km, barrio_precio_mediano, "
      f"barrio_precio_m2_mediano, ratio_cubierta_total, barrio_freq")


#-----------------Definicion de features y target------------------
FEATURES = [
    "superficie_cubierta_m2",
    "superficie_total_m2",
    "ambientes",
    "dormitorios",
    "banios",
    "antiguedad_anios",
    "antiguedad_missing",   # flag informativo
    "cochera",
    "balcon",
    "pileta",
    "tipo_enc",
    "fuente_enc",
    "anio_publicacion",
    "mes_publicacion",
    "barrio_precio_mediano",
    "barrio_precio_m2_mediano",
    "barrio_freq",
    "ratio_cubierta_total",
    "dist_centro_km",       # NaN donde no hay coords
]

TARGET = "precio_ars"

# Imputar dist_centro_km nula con mediana (para no perder ~14% de filas)
df["dist_centro_km"].fillna(df["dist_centro_km"].median(), inplace=True)

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"\nSet de modelado: {X.shape[0]} filas, {X.shape[1]} features")
print(f"Target — media: {y.mean():,.0f} | mediana: {y.median():,.0f} | std: {y.std():,.0f}")

#-----------Funcion de evalucacion cruzada------------

def cv_evaluate(model, X, y, n_splits=5, name=""):
    """
    KFold CV con 5 folds.
    Retorna dict con R², RMSE, MAE, MAPE medios y std.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    r2s, rmses, maes, mapes = [], [], [], []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        r2s.append(r2_score(y_val, preds))
        rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
        maes.append(mean_absolute_error(y_val, preds))
        # MAPE: evitamos divisiones por 0
        mask = y_val > 0
        mapes.append(np.mean(np.abs((y_val[mask] - preds[mask]) / y_val[mask])) * 100)

    results = {
        "model": name,
        "R2_mean": np.mean(r2s),   "R2_std": np.std(r2s),
        "RMSE_mean": np.mean(rmses), "RMSE_std": np.std(rmses),
        "MAE_mean": np.mean(maes),   "MAE_std": np.std(maes),
        "MAPE_mean": np.mean(mapes), "MAPE_std": np.std(mapes),
    }
    return results


#------------Modelos con MLflow-----------

import os
os.makedirs("mlruns", exist_ok=True)
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("propiedata_alquileres")

all_results = []

# ── Experimento 1: Random Forest (baseline)
print("\n[1/3] Entrenando Random Forest (baseline)...")
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

with mlflow.start_run(run_name="random_forest_baseline"):
    mlflow.log_params({
        "model": "RandomForest",
        "n_estimators": 200,
        "min_samples_leaf": 5,
        "features": len(FEATURES),
    })
    res_rf = cv_evaluate(rf, X, y, name="RandomForest")
    mlflow.log_metrics({
        "R2": res_rf["R2_mean"],
        "RMSE": res_rf["RMSE_mean"],
        "MAE": res_rf["MAE_mean"],
        "MAPE": res_rf["MAPE_mean"],
    })
    # Entrenamos el modelo final sobre todos los datos para análisis de importancia
    rf.fit(X, y)
    mlflow.sklearn.log_model(rf, "random_forest")

all_results.append(res_rf)
print(f"  RF → R²={res_rf['R2_mean']:.4f} ± {res_rf['R2_std']:.4f} | "
      f"RMSE={res_rf['RMSE_mean']:,.0f} | MAE={res_rf['MAE_mean']:,.0f} | "
      f"MAPE={res_rf['MAPE_mean']:.1f}%")

# ── Experimento 2: XGBoost con features geoespaciales (enfoque principal)
print("\n[2/3] Entrenando XGBoost con features geo...")
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)

with mlflow.start_run(run_name="xgboost_geo_features"):
    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "features": len(FEATURES),
        "geo_features": "dist_centro_km, barrio_precio_mediano, barrio_precio_m2_mediano",
    })
    res_xgb = cv_evaluate(xgb, X, y, name="XGBoost_geo")
    mlflow.log_metrics({
        "R2": res_xgb["R2_mean"],
        "RMSE": res_xgb["RMSE_mean"],
        "MAE": res_xgb["MAE_mean"],
        "MAPE": res_xgb["MAPE_mean"],
    })
    xgb.fit(X, y)
    mlflow.xgboost.log_model(xgb, "xgboost")

all_results.append(res_xgb)
print(f"  XGB → R²={res_xgb['R2_mean']:.4f} ± {res_xgb['R2_std']:.4f} | "
      f"RMSE={res_xgb['RMSE_mean']:,.0f} | MAE={res_xgb['MAE_mean']:,.0f} | "
      f"MAPE={res_xgb['MAPE_mean']:.1f}%")

# ── Experimento 3: XGBoost sin features geoespaciales (ablation)
#    Para cuantificar cuánto aportan las coords y features de barrio
FEATURES_NOGEO = [f for f in FEATURES if f not in
                  ["dist_centro_km","barrio_precio_mediano",
                   "barrio_precio_m2_mediano","barrio_freq"]]

print("\n[3/3] Entrenando XGBoost sin features geo (ablation)...")
xgb_nogeo = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)

with mlflow.start_run(run_name="xgboost_no_geo"):
    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "geo_features": "NONE",
        "features": len(FEATURES_NOGEO),
    })
    res_xgb_ng = cv_evaluate(xgb_nogeo, X[FEATURES_NOGEO], y, name="XGBoost_nogeo")
    mlflow.log_metrics({
        "R2": res_xgb_ng["R2_mean"],
        "RMSE": res_xgb_ng["RMSE_mean"],
        "MAE": res_xgb_ng["MAE_mean"],
        "MAPE": res_xgb_ng["MAPE_mean"],
    })

all_results.append(res_xgb_ng)
print(f"  XGB (sin geo) → R²={res_xgb_ng['R2_mean']:.4f} ± {res_xgb_ng['R2_std']:.4f} | "
      f"RMSE={res_xgb_ng['RMSE_mean']:,.0f} | MAE={res_xgb_ng['MAE_mean']:,.0f} | "
      f"MAPE={res_xgb_ng['MAPE_mean']:.1f}%")


#-----------Tabla comparativa de los resultados----------

print("\n" + "═"*75)
print("TABLA DE RESULTADOS — Cross-Validation 5-Fold")
print("═"*75)
print(f"{'Modelo':<25} {'R²':>8} {'±':>6} {'RMSE':>12} {'MAE':>12} {'MAPE%':>8}")
print("-"*75)
for r in all_results:
    print(f"{r['model']:<25} {r['R2_mean']:>8.4f} {r['R2_std']:>6.4f} "
          f"{r['RMSE_mean']:>12,.0f} {r['MAE_mean']:>12,.0f} {r['MAPE_mean']:>8.1f}%")
print("═"*75)

#--------Analisis de importancia de features -----------
print("\nImportancia de features — XGBoost con geo:")
importances = pd.Series(xgb.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)
for feat, imp in importances.head(10).items():
    bar = "█" * int(imp * 200)
    print(f"  {feat:<35} {imp:.4f}  {bar}")

#--------Analisis de errores-----------

# Usar último fold de XGB para análisis de errores
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
xgb_final = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    n_jobs=-1, random_state=42, verbosity=0
)
xgb_final.fit(X_tr, y_tr)
preds_te = xgb_final.predict(X_te)

errores = pd.DataFrame({
    "real": y_te.values,
    "predicho": preds_te,
    "error_abs": np.abs(y_te.values - preds_te),
    "error_pct": np.abs(y_te.values - preds_te) / y_te.values * 100,
    "tipo": df.loc[y_te.index, "tipo_inmueble"].values,
    "barrio": df.loc[y_te.index, "barrio"].values,
})

print("\n── Error por tipo de inmueble:")
print(errores.groupby("tipo")[["error_abs","error_pct"]].mean().round(0))

print("\n── Top 10 barrios con mayor error absoluto medio:")
barrio_err = errores.groupby("barrio")["error_abs"].mean().sort_values(ascending=False)
print(barrio_err.head(10).apply(lambda x: f"${x:,.0f}"))

print("\n── Distribución del error porcentual:")
for pct in [10, 25, 50, 75, 90]:
    print(f"  p{pct}: {np.percentile(errores['error_pct'], pct):.1f}%")


#---------- Conclusiones y pasos siguientes----------------

best = max(all_results, key=lambda r: r["R2_mean"])
geo_uplift = res_xgb["R2_mean"] - res_xgb_ng["R2_mean"]

print(f"""
══════════════════════════════════════════════════════
CONCLUSIONES
══════════════════════════════════════════════════════

MODELO ELEGIDO: {best['model']}
  R²   = {best['R2_mean']:.4f} ± {best['R2_std']:.4f}
  RMSE = {best['RMSE_mean']:,.0f} ARS
  MAE  = {best['MAE_mean']:,.0f} ARS
  MAPE = {best['MAPE_mean']:.1f}%

IMPACTO DE FEATURES GEOESPACIALES:
  El uplift en R² por agregar distancia al centro y
  medianas por barrio fue de +{geo_uplift:.4f} puntos.
  Esto confirma que la ubicación es un driver clave del
  precio, incluso con features espaciales simples.

POR QUÉ XGBOOST SOBRE RANDOM FOREST:
  XGBoost mostró mejor R² y menor varianza en CV.
  Su regularización (reg_alpha, reg_lambda) ayuda en
  zonas con pocos datos (barrios pequeños).

QUÉ SEGUIRÍA CON MÁS TIEMPO:
  1. Encoding de barrio más sofisticado: target encoding
     con smoothing para evitar leakage en zonas pequeñas.
  2. Features temporales: deflactar precio por inflación
     mensual para comparar precios de distintas fechas.
  3. Segmentar el modelo por tipo_inmueble — el error
     en Casas es mayor por menor volumen de datos.
  4. Optuna para búsqueda de hiperparámetros con CV anidado.
  5. LightGBM como alternativa (más rápido, similar calidad).
  6. Ver PROPUESTA.md §4.3 para el plan completo a R²=0.8.

HONESTIDAD SOBRE LIMITACIONES:
  - El dataset puede tener duplicados (mismo inmueble en
    varias plataformas) que inflan artificialmente el R².
    El matching (ver PROPUESTA.md §4.1) mejoraría la
    validez de las métricas.
  - Las medianas de barrio fueron calculadas sobre todo
    el dataset (incluyendo validación). En producción esto
    debe calcularse solo sobre train.
══════════════════════════════════════════════════════
""")